# Feature Engineering — Ames House Prices

This notebook evaluates a small set of interpretable, domain-based features. Their reusable implementation is in `src/features/engineering.py`; this notebook never becomes the production source of truth.

## Design constraints

Every feature must be available at prediction time, be logically meaningful, and be calculated identically for training and API input. No target-derived features are created, and no learned transformation is fitted in this notebook.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve()
if not (project_root / 'data').exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.data.preprocessing import load_processed_data
from src.features.engineering import FEATURE_DESCRIPTIONS, add_engineered_features

sns.set_theme(style='whitegrid', palette='deep')

base_data = load_processed_data(project_root / 'data/processed/train_clean.csv')
data = add_engineered_features(base_data)
new_features = list(FEATURE_DESCRIPTIONS)
data[new_features].head()

## Feature definitions

The following table makes each transformation explicit. This keeps the project defensible in a technical interview and helps us avoid feature names that hide unclear logic.

In [ ]:
import pandas as pd

pd.DataFrame({'feature': FEATURE_DESCRIPTIONS.keys(), 'description': FEATURE_DESCRIPTIONS.values()})

## Aggregate space features

**What we are analyzing:** total interior and outdoor area against sale price.

**Why it matters:** component fields describe the same underlying concept in fragments. Aggregates may make usable scale easier for a linear model to learn, while tree models can decide whether they add value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=data, x='TotalSF', y='SalePrice', alpha=0.65, ax=axes[0])
axes[0].set_title('Total interior square footage versus sale price')
sns.scatterplot(data=data, x='TotalOutdoorSpace', y='SalePrice', alpha=0.65, ax=axes[1])
axes[1].set_title('Total outdoor space versus sale price')
plt.tight_layout()
data[['TotalSF', 'TotalOutdoorSpace', 'SalePrice']].corr()['SalePrice'].sort_values(ascending=False)

**Interpretation:** total interior area should have a strong positive relationship with price; outdoor space may have a weaker and less linear one. We retain both as plausible inputs and will validate their real contribution through model comparison.

## Age, renovation, and bathrooms

**What we are analyzing:** whether property age, renovation status, and total bathroom utility have visible price relationships.

**Why it matters:** these features encode buyer-relevant concepts more directly than several raw fields, while remaining interpretable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=data, x='HouseAge', y='SalePrice', alpha=0.5, ax=axes[0])
axes[0].set_title('House age versus sale price')
sns.boxplot(data=data, x='HasRemodel', y='SalePrice', ax=axes[1])
axes[1].set_title('Sale price by remodel status')
sns.boxplot(data=data, x='TotalBathrooms', y='SalePrice', ax=axes[2])
axes[2].set_title('Sale price by total bathrooms')
plt.tight_layout()

## Quality and size interaction

**What we are analyzing:** quality-adjusted living area.

**Why it matters:** an extra square foot may not carry the same value in a low- and high-quality house. The interaction gives a linear model one transparent way to represent that possibility; it does not impose the same assumption on tree models.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=data, x='OverallQualGrLivArea', y='SalePrice', hue='OverallQual', palette='viridis', alpha=0.7)
plt.title('Quality × living area versus sale price')
plt.tight_layout()

data[['OverallQualGrLivArea', 'TotalBathrooms', 'HouseAge', 'YearsSinceRemodel', 'SalePrice']].corr()['SalePrice'].sort_values(ascending=False)

## Engineering conclusions

- The implemented features use only property attributes available at prediction time; there is no target leakage.
- Total interior area, house age, bathroom utility, renovation status, and the quality-size interaction are retained for Phase 5.
- The original component variables remain available. We will let comparable pipelines and held-out evaluation determine whether engineered features improve generalization.
- Missing-value handling, category encoding, and scaling remain deferred to the training-fitted pipeline in Phase 6.